In [3]:
!pip install mlflow optuna dagshub imbalanced-learn lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.5 MB/s eta 0:00:00


In [4]:
#!aws configure
import dagshub
dagshub.init(repo_owner="rohitbedse",repo_name="yt-comment-sentiment-analysis",mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d46661ba-2475-4aae-b6f1-832be97d0066&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=2cdcfe3cf48dfcea43f7695aa21bfe12e5f8b41b89f40218b5e46e5d273bc125




Accessing as rohitbedse

Initialized MLflow to track repo "rohitbedse/yt-comment-sentiment-analysis"

Repository rohitbedse/yt-comment-sentiment-analysis initialized!

In [5]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow")

In [6]:
# Set or create an experiment
mlflow.set_experiment("LightGBM HP Tuning")

2026/04/08 04:20:12 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/f732e98aa86f42b1b399f412c3966408', creation_time=1775622012304, experiment_id='10', last_update_time=1775622012304, lifecycle_stage='active', name='LightGBM HP Tuning', tags={}, trace_location=None, workspace='default'>

In [13]:
import pandas as pd

df = pd.read_csv('/content/reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [14]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report,f1_score
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import numpy as np

In [15]:
# -----------------------------
# STEP 1: Label Mapping & Drop NaNs
# -----------------------------
df = df.dropna(subset=['category'])
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# -----------------------------
# STEP 2: Show ORIGINAL label distribution
# -----------------------------
print("🔥 ORIGINAL LABELS:")
print(np.unique(df['category'], return_counts=True))

# -----------------------------
# STEP 3: TRAIN-TEST SPLIT (LEAKAGE-FREE)
# -----------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# -----------------------------
# STEP 4: TF-IDF (FIT ONLY ON TRAIN)
# -----------------------------
vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=2000)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# -----------------------------
# STEP 5: TRAIN LABEL DISTRIBUTION BEFORE SMOTE
# -----------------------------
print("\n📊 TRAIN LABELS BEFORE SMOTE:")
print(np.unique(y_train, return_counts=True))

# -----------------------------
# STEP 6: APPLY SMOTE ONLY ON TRAIN
# -----------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# -----------------------------
# STEP 7: TRAIN LABEL DISTRIBUTION AFTER SMOTE
# -----------------------------
print("\n🚀 TRAIN LABELS AFTER SMOTE:")
print(np.unique(y_train_res, return_counts=True))

# -----------------------------
# STEP 8: MLflow logging function
# -----------------------------
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigram")
        mlflow.set_tag("experiment", "No_Leakage_F1_Pipeline")

        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("resampling", "SMOTE")
        mlflow.log_param("vectorizer", "TF-IDF Trigram")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        mlflow.sklearn.log_model(model, f"{model_name}_model")

# -----------------------------
# STEP 9: OPTUNA OBJECTIVE FUNCTION (F1-MACRO)
# -----------------------------
def objective_lightgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state": 42
    }

    model = LGBMClassifier(**params)

    # Split training into train/validation for Optuna
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_res, y_train_res,
        test_size=0.2,
        random_state=42,
        stratify=y_train_res
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred, average='macro')

# -----------------------------
# STEP 10: RUN OPTUNA EXPERIMENT
# -----------------------------
def run_optuna_experiment(n_trials=100):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=n_trials)

    print("\n🏆 BEST PARAMS:", study.best_params)

    best_model = LGBMClassifier(**study.best_params, random_state=42)
    log_mlflow("LightGBM", best_model, X_train_res, X_test, y_train_res, y_test)

    # Optional: Plot parameter importance
    import matplotlib.pyplot as plt
    import optuna.visualization as vis
    fig1 = vis.plot_param_importances(study)
    fig1.show()
    fig2 = vis.plot_optimization_history(study)
    fig2.show()

# -----------------------------
# RUN THE PIPELINE
# -----------------------------
run_optuna_experiment()

🔥 ORIGINAL LABELS:
(array([0, 1, 2]), array([12644, 15770,  8248]))

📊 TRAIN LABELS BEFORE SMOTE:
(array([0, 1, 2]), array([10115, 12616,  6598]))


[I 2026-04-08 05:04:43,974] A new study created in memory with name: no-name-27bb8802-98be-4345-a267-ad31cccf5305



🚀 TRAIN LABELS AFTER SMOTE:
(array([0, 1, 2]), array([12616, 12616, 12616]))
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.216211 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 84423
[LightGBM] [Info] Number of data points in the train set: 30278, number of used features: 977
[LightGBM] [Info] Start training from score -1.098579
[LightGBM] [Info] Start training from score -1.098579
[LightGBM] [Info] Start training from score -1.098678
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

[I 2026-04-08 05:04:52,591] Trial 0 finished with value: 0.6250045898493651 and parameters: {'n_estimators': 187, 'learning_rate': 0.007105254559188081, 'max_depth': 5, 'num_leaves': 73, 'min_child_samples': 95, 'colsample_bytree': 0.6589038674321022, 'subsample': 0.8364339094107611, 'reg_alpha': 0.0002624211354683743, 'reg_lambda': 2.9548772918321835}. Best is trial 0 with value: 0.6250045898493651.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.201669 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 84912
[LightGBM] [Info] Number of data points in the train set: 30278, number of used features: 992
[LightGBM] [Info] Start training from score -1.098579
[LightGBM] [Info] Start training from score -1.098579
[LightGBM] [Info] Start training from score -1.098678
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.853689 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 106316
[LightGBM] [Info] Number of data points in the train set: 30278, number of used features: 1948
[LightGBM] [Info] Start training from score -1.098579
[LightGBM] [Info] Start training from score -1.098579
[LightGBM] [Info] Start training from score -1.098678
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

[I 2026-04-08 05:48:43,950] Trial 99 finished with value: 0.8214356591965792 and parameters: {'n_estimators': 255, 'learning_rate': 0.06241119005524533, 'max_depth': 10, 'num_leaves': 20, 'min_child_samples': 14, 'colsample_bytree': 0.7286565184272424, 'subsample': 0.6969378623478286, 'reg_alpha': 0.0001814714920041537, 'reg_lambda': 0.7110852848620597}. Best is trial 72 with value: 0.8360103965870103.



🏆 BEST PARAMS: {'n_estimators': 280, 'learning_rate': 0.08373507882897219, 'max_depth': 10, 'num_leaves': 88, 'min_child_samples': 16, 'colsample_bytree': 0.8460502953689186, 'subsample': 0.7095305073802686, 'reg_alpha': 0.0024901271471122915, 'reg_lambda': 0.00407778445954906}
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.637287 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 127247
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 1953
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names

2026/04/08 05:49:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 05:50:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM_SMOTE_TFIDF_Trigram at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/10/runs/2d9ae35252d04104a26d840f85bebacf
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/10


NameError: name 'best_model' is not defined